# Tech Challenge - Fase 1 (EXTRA: Visão Computacional)
## Detecção de Pneumonia em Radiografias de Tórax usando CNN (Keras / TensorFlow)

Este notebook executa o treinamento e a avaliação de uma Redes Neurais Convolucionais (CNN) para a classificação de imagens radiográficas em duas classes: **NORMAL** e **PNEUMONIA**.

MONTAR GOOGLE DRIVE E CRIAR DIRETÓRIOS

In [ ]:
# 1. Montagem do Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Configuração de diretórios de entrada e saída
DRIVE_PATH = '/content/drive/MyDrive'
RESULTS_DIR = os.path.join(DRIVE_PATH, 'pneumonia_cnn_resultados')
MODEL_SAVE_PATH = os.path.join(RESULTS_DIR, 'pneumonia_cnn_best.keras')

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Diretório de resultados pronto em: {RESULTS_DIR}')

BIBLIOTECAS E REPRODUÇÕES ALEATÓRIAS

In [ ]:
# 2. Importação das Bibliotecas Necessárias
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Definindo sementes para reproducibilidade
np.random.seed(42)
tf.random.set_seed(42)
print('TensorFlow Version:', tf.__version__)

VALIDANDO EXISTÊNCIA DE DIRETÓRIOS

In [ ]:
# 3. Localização do Dataset no Drive
DATASET_PATH = os.path.join(DRIVE_PATH, 'datasets', 'chest_xray')

TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
VAL_DIR = os.path.join(DATASET_PATH, 'val') if os.path.exists(os.path.join(DATASET_PATH, 'val')) else os.path.join(DATASET_PATH, 'test')
TEST_DIR = os.path.join(DATASET_PATH, 'test')

# Validação se os diretórios existem
print(f'Treino existe? {os.path.exists(TRAIN_DIR)} -> {TRAIN_DIR}')
print(f'Validação existe? {os.path.exists(VAL_DIR)} -> {VAL_DIR}')
print(f'Teste existe? {os.path.exists(TEST_DIR)} -> {TEST_DIR}')

if not os.path.exists(TRAIN_DIR):
    raise FileNotFoundError(f"A pasta 'train' não foi encontrada em '{TRAIN_DIR}'. Verifique o caminho.")

PRÉ-PROCESSAMENTO

In [ ]:
# 4. Pré-processamento e Data Augmentation
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

val_generator = test_val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_generator = test_val_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

CONSTRUÇÃO DA ARQUITETURA CNN

In [ ]:
# 5. Construção da Arquitetura da CNN
model = models.Sequential([
    layers.Input(shape=(150, 150, 3)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.Precision(name='precision')]
)

model.summary()

TREINAMENTO DE MODELO

In [ ]:
# 6. Callbacks e Treinamento do Modelo
callbacks = [
    ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        verbose=1
    )
]

EPOCHS = 25
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks
)

GRÁFICOS (ACURÁCIA E PERDA)

In [ ]:
# 7. Curvas de Aprendizado (Acurácia e Loss)
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Treino')
plt.plot(history.history['val_accuracy'], label='Validação')
plt.title('Acurácia por Época')
plt.xlabel('Épocas')
plt.ylabel('Acurácia')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.title('Perda (Loss) por Época')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

acc_loss_plot_path = os.path.join(RESULTS_DIR, 'curvas_treinamento.png')
plt.savefig(acc_loss_plot_path)
plt.show()
print(f'Gráfico salvo em: {acc_loss_plot_path}')

AVALIAÇÃO FINAL E PERSISTÊNCIA

In [ ]:
# 8. Avaliação Final nos Dados de Teste
best_model = keras.models.load_model(MODEL_SAVE_PATH)

test_loss, test_acc, test_recall, test_precision = best_model.evaluate(test_generator)
f1_score = 2 * (test_precision * test_recall) / (test_precision + test_recall + 1e-7)

print(f'--- RESULTADOS NOS DADOS DE TESTE ---')
print(f'Acurácia: {test_acc:.4f}')
print(f'Recall: {test_recall:.4f}')
print(f'Precisão: {test_precision:.4f}')
print(f'F1-Score: {f1_score:.4f}')

# Matriz de Confusão
y_pred_probs = best_model.predict(test_generator)
y_pred = (y_pred_probs > 0.5).astype(int).ravel()
y_true = test_generator.classes

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['NORMAL', 'PNEUMONIA'], yticklabels=['NORMAL', 'PNEUMONIA'])
plt.title('Matriz de Confusão')
plt.xlabel('Predito')
plt.ylabel('Real')

cm_plot_path = os.path.join(RESULTS_DIR, 'matriz_confusao.png')
plt.savefig(cm_plot_path)
plt.show()

# Salvar Relatório de Classificação em TXT
report_text = classification_report(y_true, y_pred, target_names=['NORMAL', 'PNEUMONIA'])
report_file_path = os.path.join(RESULTS_DIR, 'relatorio_classificacao.txt')
with open(report_file_path, 'w') as f:
    f.write('--- RELATÓRIO DE AVALIAÇÃO CNN - PNEUMONIA ---\n\n')
    f.write(f'Acurácia Teste: {test_acc:.4f}\n')
    f.write(f'Recall Teste: {test_recall:.4f}\n')
    f.write(f'Precisão Teste: {test_precision:.4f}\n')
    f.write(f'F1-Score Teste: {f1_score:.4f}\n\n')
    f.write(report_text)

print(f'Relatório completo salvo em: {report_file_path}')